In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# improt the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from glob import glob
import seaborn as sns
from tabulate import tabulate
import random
import os
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
!unzip /content/data.zip -d data

In [ ]:
# Map the images from a folder to form a DataFrame
def get_all_images_from_subdirectory_to_dataframe(path):
  # Get all file paths of '.jpg' files within the specified path and its subdirectories
  configfiles = [os.path.join(dirpath, f)
      for dirpath, dirnames, files in os.walk(path)
      for f in files if f.endswith('.jpg')]

  # Create a list of tuples containing image names and their corresponding paths
  images_list = [(i.split("/")[-1].split(".")[0], i.split("/")[-1], i, i.split("/")[-1].split(".")[-1]) for i in configfiles]

  # Return the list of images
  return images_list

In [ ]:
# Create a DataFrame from a list of images and their paths
path = '/content/data'
df_list = get_all_images_from_subdirectory_to_dataframe(path)

# Construct a DataFrame with columns 'Image_id' and 'image_path'
df = pd.DataFrame(data=df_list, columns=['class', 'image_name', 'image_path', 'ext'])

# Display the first few rows of the DataFrame
df.head()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def plot_random_images(df, total_image=2):
    fig, axes = plt.subplots(1, total_image, figsize=(14, 2))
    images_data = list(zip(df['image_path'], df['class']))
    samples = random.sample(images_data, total_image)

    for ax, (image_path, label) in zip(axes, samples):
        image = Image.open(image_path)
        ax.set_axis_off()
        ax.imshow(image, cmap='binary')
        ax.set_title(f'{label}')

    plt.show()


In [ ]:
plot_random_images(df, 5)
plot_random_images(df, 5)
plot_random_images(df, 5)

In [ ]:
# Calculate the count of each class
class_counts = df['class'].value_counts()
total_classes = len(df)

# Calculate the percentage of each class
class_percentages = (class_counts / total_classes) * 100

# Print the percentage of each class with count
for class_name, count in class_counts.items():
    percentage = class_percentages[class_name]
    print(f"Class: {class_name}\t\t\t| Count: {count}\t\t| Percentage: {percentage:.2f}%")

In [ ]:
def class_distribution(df_final, col):
    plt.figure(figsize=(17, 7))

    plt.subplot(1, 2, 1)

    colors = ['red', 'green', 'blue', 'magenta', 'orange', 'purple', 'cyan', 'yellow', 'brown']
    ax = df_final[col].value_counts().plot(kind='bar', color=colors)

    plt.xlabel('Category', fontsize=16)
    plt.ylabel('Frequency of Class', fontsize=16)
    plt.xticks(size=12)
    plt.yticks(size=12)
    plt.title(f'Frequency Distribution of Class', fontsize=18)
    for p in ax.patches:
        ax.annotate(str(p.get_height()), (p.get_x() * 1.005, p.get_height() * 1.01), size=15)

    plt.subplot(1, 2, 2)

    df_final[col].value_counts().plot.pie(autopct='%1.2f%%', shadow=True, colors=colors,
                                          textprops={'fontsize': 15, 'color': 'white'})
    plt.ylabel('Target', fontsize=16)
    plt.title(f'Proportional Distribution of Class', fontsize=18)
    plt.legend(bbox_to_anchor=(1, 1), loc='upper left')
    plt.show()

In [ ]:
class_distribution(df, 'class')

In [ ]:
cv2.imread(df['image_path'][0]).shape

In [ ]:
from sklearn.preprocessing import LabelEncoder

# convert the categorical values to numeric values
labelEncoder = LabelEncoder()
df['class'] = labelEncoder.fit_transform(df['class'])

In [ ]:
labelEncoder.classes_

In [ ]:
# Create X and Y arrays from the DataFrame
height = 150
width = 150
channel = 3

def get_X_Y(df):
    images, labels = [], []
    # Iterate over image paths
    for img_path, target in tqdm(zip(df['image_path'],df['class'])):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (height, width))  # Unify shape of all the images.

        images.append(img)
        labels.append(target)

    # Convert images and labels to numpy arrays
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    return images, labels

In [ ]:
# Get X and y arrays using the get_X_Y function
X, y = get_X_Y(df)

In [ ]:
X.shape, y.shape

In [ ]:
# Normalise all the images
def normalize_images(images):
  # convert into float32
  images = images.astype('float32')
  images /= 255.0
  return images

In [ ]:
# Normalize the images in X
X_norm = normalize_images(X)
print("done.")

In [ ]:
import random
import tensorflow as tf
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the data into training and remaining (validation + test) sets
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.35, random_state=0, stratify=y)
# Printing the shapes of the datasets
print("Training Data:", X_train.shape)
print("Test Data:", X_test.shape)

In [ ]:
print("Training Splition Data")
print(pd.Series(y_train).astype(int).value_counts())
print()
print("Testing Splition Data")
print(pd.Series(y_test).astype(int).value_counts())

In [ ]:
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNet

## **MobileNet + Random Forest**

In [ ]:
# Load the pre-trained model
model = MobileNet(weights='imagenet', include_top=False, input_shape=(height, width, channel))

# Freeze the pre-trained layers to prevent their weights from being updated during training
for layer in model.layers:
    layer.trainable = False

In [ ]:
# model.summary()

In [ ]:
feature_extractor = model.predict(X_train, verbose=0)
features = feature_extractor.reshape(feature_extractor.shape[0], -1)
# Get the shape of the feature_extractor array
features.shape

In [ ]:
X_for_rf = features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
rf = RandomForestClassifier(random_state=42)
rf.fit(X_for_rf, y_train)

In [ ]:
X_test_feature = model.predict(X_test, verbose=0)
X_test_features = X_test_feature.reshape(X_test_feature.shape[0], -1)
X_test_features.shape

In [ ]:
prediction_RF = rf.predict(X_test_features)

In [ ]:
def GetPredictionTestingData(model):
  # Perform predictions on the validation set
  y_pred = model.predict(X_test)
  y_pred = np.argmax(np.round(y_pred), axis=1)
  return y_pred

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay

In [ ]:
def results(model_name, y_pred, y_test):
    target_names = ["{}".format(Classes[i]) for i in range(len(Classes))] # Define target names for classification report
    accuracy = round(accuracy_score(y_pred, y_test)*100,4)
    precision = round(precision_score(y_pred, y_test, average='macro')*100,4)
    recall = round(recall_score(y_pred, y_test, average='macro')*100,4)
    f1_scr = round(f1_score(y_pred, y_test, average='macro')*100,4)


    print("\nAccuracy: {}%".format(accuracy))
    print("Precision: {}%".format(precision))
    print("Recall: {}%".format(recall))
    print("F1-Score: {}%".format(f1_scr))
    print()
    print("Classification Report:")
    print(classification_report(y_pred, y_test, target_names=target_names))
    print()
    print("Confusion Matrix:")
    fig, ax = plt.subplots(figsize=(7,5))
    ConfusionMatrixDisplay.from_predictions(y_pred, y_test,
                                            ax=ax,
                                            display_labels=target_names,
                                            xticks_rotation='vertical')
    plt.show()

    return {
        'Model':model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1_scr
    }

In [ ]:
def GetModelResultsInDataFrame(res):
  # Convert the dictionary to a DataFrame
  return pd.DataFrame.from_dict([res]).set_index('Model')

In [ ]:
# Get the list of Classes from a label encoder
Classes = list(labelEncoder.classes_)
Classes

In [ ]:
cnn_rf_res = GetModelResultsInDataFrame(results("CNN+RF", prediction_RF, y_test))
cnn_rf_res

In [ ]:
# Initialize parameters for bootstrapping
num_bootstrap_samples = 100  # Number of bootstrap samples
ensemble_accuracies = []

# Perform bootstrapping
for _ in range(num_bootstrap_samples):
    # Create a bootstrap sample
    bootstrap_indices = np.random.choice(len(X_test_features), len(X_test_features), replace=True)
    bootstrap_X = X_test_features[bootstrap_indices]
    bootstrap_y = y_test[bootstrap_indices]

    # Predict class labels using the bagging ensemble
    bootstrap_y_pred =  rf.predict(bootstrap_X)

    # Calculate accuracy for the bootstrap sample
    bootstrap_accuracy = accuracy_score(bootstrap_y, bootstrap_y_pred)
    ensemble_accuracies.append(bootstrap_accuracy)

# Calculate mean and confidence interval for ensemble accuracy
mean_accuracy = np.mean(ensemble_accuracies)
lower_bound = np.percentile(ensemble_accuracies, 2.5)
upper_bound = np.percentile(ensemble_accuracies, 97.5)

# Print uncertainty quantification results
print(f"Ensemble Mean Accuracy: {mean_accuracy:.4f}")
print(f"Confidence Interval: {lower_bound:.4f} - {upper_bound:.4f}")

In [ ]:
y_prob =  rf.predict_proba(X_test_features)  # Probability estimates for each class

# Assuming you want to visualize uncertainty by plotting probability distributions
plt.figure(figsize=(12, 6))
for i, class_name in enumerate(Classes):
    class_probs = y_prob[:, i]
    plt.subplot(2, 5, i + 1)
    plt.hist(class_probs, bins=20, alpha=0.7, color='blue')
    plt.title(f'Class {class_name} Probabilities')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots to display images
num_classes = len(Classes)
num_images_per_class = 4  # You can adjust this as per your preference
fig, axes = plt.subplots(num_classes, num_images_per_class, figsize=(12, 12))

for i, class_name in enumerate(Classes):
    # Get indices of images for the current class
    class_indices = np.where(y_test == i)[0]

    # Randomly select num_images_per_class correct and incorrect predictions
    np.random.shuffle(class_indices)
    selected_correct_indices = class_indices[(y_test[class_indices] == prediction_RF[class_indices])][:num_images_per_class // 2]
    selected_incorrect_indices = class_indices[(y_test[class_indices] != prediction_RF[class_indices])][:num_images_per_class // 2]

    # Plot correct predictions
    for j, correct_idx in enumerate(selected_correct_indices):
        ax = axes[i, j * 2]
        ax.imshow(X_test[correct_idx], cmap='gray')
        ax.axis('off')
        ax.set_title(f"Correct: {Classes[i]}")

    # Plot incorrect predictions
    for j, incorrect_idx in enumerate(selected_incorrect_indices):
        ax = axes[i, j * 2 + 1]
        ax.imshow(X_test[incorrect_idx], cmap='gray')
        ax.axis('off')
        predicted_label = int(prediction_RF[incorrect_idx])  # Convert to integer
        ax.set_title(f"Predicted: {Classes[predicted_label]}, Actual: {Classes[i]}")

# Adjust spacing between subplots
plt.tight_layout()
plt.show()